# Red de categorías: co-ofertas Categoria–Categoria

Nodo = VERTICAL (la categoría de Mercado Libre).

Arista = dos VERTICAL que aparecen juntas en el mismo evento de oferta.

Pesos posibles en la arista:

*      cooffers_cat = nº de eventos en los que esas dos categorías aparecen juntas.

*      joint_qty_cat = ventas conjuntas asociadas al par de categorías.

In [2]:
import pandas as pd
from itertools import combinations
import networkx as nx

df = pd.read_csv("/media/paulina/TOSHIBA EXT/2024/Codigos/MercadoLibre/arquivos/ofertas_relampago.csv", encoding="latin1")

df["OFFER_START_DTTM"] = pd.to_datetime(df["OFFER_START_DTTM"])
df["OFFER_FINISH_DTTM"] = pd.to_datetime(df["OFFER_FINISH_DTTM"])

#ID de evento de oferta
df["EVENT_ID"] = df["OFFER_START_DTTM"]

# Rellenar NaN de SOLD_QUANTITY con 0 para no romper los cálculos
df["SOLD_QUANTITY"] = df["SOLD_QUANTITY"].fillna(0)

In [3]:
import pandas as pd
import networkx as nx
from itertools import combinations

def crear_df_coofertas_verticales(df, event_cols=None):
    """
    Crea un DataFrame de edges entre categorías (VERTICAL–VERTICAL)
    basados en co-ofertas en los mismos eventos.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame original de ofertas relámpago.
    event_cols : list[str], opcional
        Columnas que definen un "evento de oferta".
        Por defecto: ["OFFER_START_DTTM", "OFFER_TYPE"].

    Retorna
    -------
    edges_vert : pd.DataFrame
        columnas:
        - vert_i
        - vert_j
        - cooffers   (nº de eventos en los que co-ocurren)
        - joint_qty  (ventas conjuntas acumuladas entre categorías)
    """
    if event_cols is None:
        event_cols = ["OFFER_START_DTTM", "OFFER_TYPE"]

    # Asegurar que SOLD_QUANTITY no tenga NaN
    df = df.copy()
    df["SOLD_QUANTITY"] = df["SOLD_QUANTITY"].fillna(0)

    # Nos quedamos con columnas necesarias
    cols = event_cols + ["VERTICAL", "SOLD_QUANTITY"]
    df_event = df[cols].copy()

    edge_rows = []

    # Agrupar por evento (combinación de columnas event_cols)
    for _, group in df_event.groupby(event_cols):
        # Agregar ventas por categoría dentro del evento
        vert_event = (
            group.groupby("VERTICAL")["SOLD_QUANTITY"]
                 .sum()
                 .reset_index()
        )

        verticals = vert_event["VERTICAL"].values
        qty_by_vert = vert_event.set_index("VERTICAL")["SOLD_QUANTITY"]

        # Si sólo hay una categoría en el evento, no hay aristas
        if len(verticals) < 2:
            continue

        # Todas las combinaciones de categorías dentro del evento
        for a, b in combinations(verticals, 2):
            joint_qty = float(qty_by_vert[a] + qty_by_vert[b])
            edge_rows.append((a, b, 1, joint_qty))

    # Agregar sobre todos los eventos
    edges_vert = (
        pd.DataFrame(edge_rows, columns=["vert_i", "vert_j", "cooffers", "joint_qty"])
        .groupby(["vert_i", "vert_j"], as_index=False)
        .agg(
            cooffers=("cooffers", "sum"),
            joint_qty=("joint_qty", "sum")
        )
    )

    return edges_vert


edges_vert = crear_df_coofertas_verticales(df)
#edges_vert.to_csv("edges_cooffers_VERTICAL_VERTICAL.csv", index=False)

In [4]:
edges_vert

,vert_i,vert_j,cooffers,joint_qty
0,ACC,APP & SPORTS,203,27435.0
1,ACC,BEAUTY & HEALTH,210,60671.0
2,ACC,CE,200,23003.0
3,ACC,CPG,191,8675.0
4,ACC,ENTERTAINMENT,156,4688.0
5,ACC,HOME & INDUSTRY,207,27287.0
6,ACC,OTHERS,111,3322.0
7,ACC,T & B,190,7074.0
8,APP & SPORTS,BEAUTY & HEALTH,458,176754.0
9,APP & SPORTS,CE,418,45915.0


In [5]:
def crear_df_nodos_verticales(df):
    """
    Crea un DataFrame de nodos de categoría (VERTICAL) con atributos agregados.

    Devuelve columnas como:
    - VERTICAL
    - num_products
    - mean_stock, total_stock
    - mean_sold, total_sold
    - stockout_rate
    """
    df = df.copy()
    df["SOLD_QUANTITY"] = df["SOLD_QUANTITY"].fillna(0)

    nodos_vert = (
        df.groupby("VERTICAL")
          .agg(
              num_products=("DOMAIN_ID", "nunique"),
              mean_stock=("INVOLVED_STOCK", "mean"),
              total_stock=("INVOLVED_STOCK", "sum"),
              mean_sold=("SOLD_QUANTITY", "mean"),
              total_sold=("SOLD_QUANTITY", "sum"),
              stockout_rate=("REMAINING_STOCK_AFTER_END", lambda x: (x < 0).mean())
          )
          .reset_index()
    )

    return nodos_vert


nodos_vert = crear_df_nodos_verticales(df)
#nodos_vert.to_csv("nodos_VERTICAL.csv", index=False)

In [6]:
nodos_vert

,VERTICAL,num_products,mean_stock,total_stock,mean_sold,total_sold,stockout_rate
0,ACC,84,9.565099,24611,2.003887,5156.0,0.033813
1,APP & SPORTS,191,18.122347,239958,1.937769,25658.0,0.016237
2,BEAUTY & HEALTH,146,127.335524,910449,24.914825,178141.0,0.041538
3,CE,185,24.500753,211417,2.530189,21833.0,0.021439
4,CPG,94,14.723287,39959,2.098747,5696.0,0.039425
5,ENTERTAINMENT,17,6.032381,3167,0.680000,357.0,0.000000
6,HOME & INDUSTRY,407,20.907002,230207,2.466715,27161.0,0.031151
7,OTHERS,39,10.394265,2900,1.279570,357.0,0.010753
8,T & B,103,16.695122,43808,0.895198,2349.0,0.013720


In [7]:
def crear_red_coofertas_verticales(edges_vert, nodos_vert, weight_col="cooffers"):
    """
    Crea la red de co-ofertas entre categorías (VERTICAL–VERTICAL).

    - edges_vert: DataFrame con columnas vert_i, vert_j, cooffers, joint_qty
    - nodos_vert: DataFrame con atributos agregados por VERTICAL
    - weight_col: qué usar como atributo 'weight' en la arista
                  ("cooffers" para nº de eventos, "joint_qty" para ventas conjuntas)

    Retorna:
    - G_vert: networkx.Graph
    """
    if weight_col not in ["cooffers", "joint_qty"]:
        raise ValueError("weight_col debe ser 'cooffers' o 'joint_qty'")

    G_vert = nx.Graph()

    # Añadir nodos y atributos
    for _, row in nodos_vert.iterrows():
        G_vert.add_node(
            row["VERTICAL"],
            num_products=int(row["num_products"]),
            mean_stock=float(row["mean_stock"]) if pd.notna(row["mean_stock"]) else None,
            total_stock=float(row["total_stock"]) if pd.notna(row["total_stock"]) else None,
            mean_sold=float(row["mean_sold"]) if pd.notna(row["mean_sold"]) else None,
            total_sold=float(row["total_sold"]) if pd.notna(row["total_sold"]) else None,
            stockout_rate=float(row["stockout_rate"]) if pd.notna(row["stockout_rate"]) else None,
        )

    # Añadir aristas con pesos
    for _, row in edges_vert.iterrows():
        G_vert.add_edge(
            row["vert_i"],
            row["vert_j"],
            weight=float(row[weight_col]),        # peso principal para layouts/algoritmos
            cooffers=int(row["cooffers"]),        # nº de eventos
            joint_qty=float(row["joint_qty"])     # ventas conjuntas
        )

    return G_vert


G_vert_events = crear_red_coofertas_verticales(edges_vert, nodos_vert, weight_col="cooffers")
print(G_vert_events.number_of_nodes(), "nodos de categoría")
print(G_vert_events.number_of_edges(), "aristas entre categorías")

nx.write_gexf(G_vert_events, "Red_VERTICAL_VERTICAL_cooffers.gexf")

G_vert_sales = crear_red_coofertas_verticales(edges_vert, nodos_vert, weight_col="joint_qty")
nx.write_gexf(G_vert_sales, "Red_VERTICAL_VERTICAL_jointqty.gexf")

9 nodos de categoría
36 aristas entre categorías
